# Module 4.1: University OneDrive Migration & Self-Hosted Web File Management (`rclone` + FileBrowser Quantum)

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 4: File Management & Cloud De-Googling/Offboarding

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Architect a clean, verifiable migration pipeline to rescue coursework, research data, projects, and personal files from an expiring **university-sponsored Microsoft 365 OneDrive** onto your redundant **UniFi `UNAS 2` RAID 1** array.
2. Navigate real-world **Microsoft Entra ID (Education tenant)** third-party OAuth blocks (*"Need admin approval"*) using both headless `rclone` OneDrive pulls (Path A) and an IT-approved workstation/SMB bridge with cryptographic integrity checking (Path B).
3. Compare open-source OneDrive replacements (**FileBrowser Quantum**, **Nextcloud**, **Syncthing**, and **UniFi Drive**) and understand why **FileBrowser Quantum** directly over your existing NFS mount avoids database duplication and overhead.
4. Deploy **FileBrowser Quantum (`ghcr.io/gtsteffaniak/filebrowser:stable`)** on `homelab-gpu` (`10.0.0.201:8900`) providing a fast, browser-based OneDrive/Google Drive experience directly over your `UNAS 2` RAID 1 storage (accessible anywhere over Tailscale).

## 4.1.1 Conceptual Background: Escaping University OneDrive Lock-In & Choosing an Open-Source File Manager

Graduating from university starts a silent countdown timer on your institutional Microsoft 365 Education license: typically **30 to 180 days** after graduation, IT revokes your OneDrive quota (often 1 TB–5 TB) and permanently purges your `.edu` mailbox and SharePoint/OneDrive files.

### 1. Why Migrating University OneDrive Is Tricky (The Microsoft Entra ID Consent Trap)
Many guides simply say *"run `rclone config`, select OneDrive, and log in!"* In practice, university IT departments aggressively lock down **Microsoft Entra ID** (formerly Azure AD) so students cannot grant `Files.Read.All` OAuth consent to unverified third-party apps like `rclone`, displaying a red **`Need admin approval`** banner.

Instead of getting stuck, we engineer **two proven migration paths** directly into your `UNAS 2` RAID 1 share (`/mnt/unas_media/University_OneDrive_Archive`):

```mermaid
flowchart LR
    UniOD["University OneDrive<br>(Microsoft 365 Education<br>Expiring Post-Graduation)"]
    
    UniOD -->|"Path A (If Tenant Allows OAuth):<br>Headless rclone copy on homelab-gpu"| UNAS["UniFi UNAS 2 (RAID 1)<br>/mnt/unas_media/<br>University_OneDrive_Archive"]
    
    UniOD -->|"Path B (If Entra ID Blocks OAuth):<br>Official OneDrive App / Web Batch<br>on Laptop -> SMB / rsync"| UNAS
    
    UNAS -->|"NFSv3 Mount (/mnt/unas_media)"| FB["FileBrowser Quantum<br>http://10.0.0.201:8900<br>(Self-Hosted Web File Manager)"]
    UNAS -. "Module 4.2: Copy PDFs/Office Docs" .-> Paperless["Paperless-ngx OCR Vault<br>http://10.0.0.201:8000"]
    UNAS -. "Module 6.2: Encrypted Cloud Copy" .-> GDrive["5 TB Google Drive<br>(Included w/ Gemini Pro)"]
```

### 2. Choosing Your Open-Source OneDrive Replacement

Once your files live safely on `UNAS 2`, you want a sleek web interface to browse, preview, upload, and share files from any browser or phone over Tailscale. Here is how the top open-source options compare on your hardware:

| Platform | Resource Footprint | Storage Model | Best Fit in Your Homelab |
| :--- | :---: | :--- | :--- |
| **FileBrowser Quantum** (`gtsteffaniak/filebrowser`) | **Ultra-light (~60 MB RAM)** | Reads your existing `UNAS 2` `/mnt/unas_media` folders **directly** (zero duplicate storage or hidden proprietary directories!) | **🥇 Primary Web File Manager (Deployed Here):** Instant Google Drive/OneDrive-like web UI, drag-and-drop uploads, PDF/video/image previews, share links, and direct 1:1 visibility of your SMB/NFS folders. |
| **UniFi Drive Web/Mobile** *(Built into UNAS 2)* | 0 MB extra (runs on `UNAS 2`) | Native UniFi Drive shares | Great built-in complement for quick mobile access and SMB/Time Machine sharing. |
| **Syncthing** *(Optional Companion)* | Light (~100 MB RAM) | Continuous peer-to-peer local folder sync | **Optional Add-On** if you want a live, offline-capable auto-syncing folder on your laptop (just like the OneDrive desktop sync client!) mirrored automatically into `/mnt/unas_media/Sync`. |
| **Nextcloud AIO** | Heavy (~2–4 GB RAM + PHP/DB/Redis) | Owns its own internal data tree; requires periodic major upgrades | Overkill when you **already** have **Immich** (photos/videos), **Paperless-ngx** (OCR docs), and **FileBrowser** (files)—avoid unnecessary maintenance bloat unless you specifically need self-hosted CalDAV/CardDAV or browser-based Word/Excel editing! |

## 4.1.2 Step-by-Step Laboratory Guide

### Task 4.1.1: Install `rclone` on `homelab-gpu` & Migrate Your University OneDrive to `UNAS 2`
1. SSH into `homelab-gpu` (`ssh homelab-gpu`) and install the latest upstream `rclone` binary (always prefer upstream `rclone.org/install.sh` over older Ubuntu apt packages so Microsoft Graph and Google Drive APIs are 100% current):
   ```bash
   curl -fsSL https://rclone.org/install.sh | sudo bash
   rclone version
   ```
2. Create a clean destination archive folder on your `UNAS 2` RAID 1 share:
   ```bash
   mkdir -p /mnt/unas_media/University_OneDrive_Archive
   mkdir -p /mnt/unas_media/Personal_Files
   ```
3. **Attempt Path A (Direct Headless `rclone` Migration — Fastest if Your University Allows OAuth):**
   - On your **personal workstation** (which has a web browser), install `rclone` (`brew install rclone` on macOS or `winget install Rclone.Rclone` on Windows) and run:
     ```bash
     rclone authorize "onedrive"
     ```
   - Log in with your university `.edu` Microsoft account in the browser window that pops up.
   - **If Microsoft grants the token:** Copy the `{"access_token":...}` JSON block printed in your terminal, then on **`homelab-gpu`** run `rclone config`:
     - `n` (New remote) $\rightarrow$ Name: `uni-onedrive` $\rightarrow$ Storage: `onedrive`
     - Leave `client_id` & `client_secret` blank $\rightarrow$ `Edit advanced config?` `n` $\rightarrow$ `Use web browser to automatically authenticate?` **`n`** $\rightarrow$ Paste the JSON token $\rightarrow$ Select **`1` (OneDrive Personal or Business)** $\rightarrow$ Select your university drive!
   - Test listing your root folders and launch a resumable, verified pull directly onto `UNAS 2` inside `tmux` (so closing your SSH window never interrupts a multi-hour transfer):
     ```bash
     rclone lsd uni-onedrive:

     sudo apt install -y tmux
     tmux new -s onedrive-migration
     # Run inside tmux:
     rclone copy uni-onedrive: /mnt/unas_media/University_OneDrive_Archive \
         --transfers 4 \
         --checkers 8 \
         --tpslimit 4 \
         --retries 5 \
         --stats 15s \
         --progress \
         --log-file /tmp/onedrive-migration.log
     ```
     *(Tip: Notice `--tpslimit 4` prevents Microsoft Graph API `429 Too Many Requests` throttling!)*

4. **Fallback — Path B (If University Entra ID Shows `"Need admin approval"`):**
   Do not waste weeks emailing university IT for an OAuth exception! Use the IT-approved official Microsoft OneDrive sync path cleanly:
   - **Option B1 (Official OneDrive Desktop Client):** Sign into the official Microsoft OneDrive app on your laptop, right-click your OneDrive root folder $\rightarrow$ **Always keep on this device** (to download all cloud-only placeholders locally), then copy directly onto your `UNAS 2` SMB share (`smb://10.0.0.202/MediaArchive/University_OneDrive_Archive`) or push via `rsync` from your workstation over wired Ethernet:
     ```bash
     # From your macOS/Linux workstation once OneDrive has hydrated locally:
     rsync -avh --progress --stats \
         "$HOME/Library/CloudStorage/OneDrive-YourUniversity/" \
         pi@10.0.0.201:/mnt/unas_media/University_OneDrive_Archive/
     ```
   - **Option B2 (OneDrive Web Portal Batch Download):** Open `onedrive.com` in your browser, select top-level folders in batches (<10–20 GB each), click **Download** (`.zip`), move the zips to `/mnt/unas_media/University_OneDrive_Archive/`, and extract directly on `homelab-gpu` using `unzip`.

5. **Post-Migration Integrity Verification (Before University Account Expires!):**
   On `homelab-gpu`, verify total migrated size, file count, and check for zero-byte corrupted placeholder files:
   ```bash
   du -sh /mnt/unas_media/University_OneDrive_Archive
   find /mnt/unas_media/University_OneDrive_Archive -type f | wc -l
   # Check if any 0-byte stub/corrupt files slipped through:
   find /mnt/unas_media/University_OneDrive_Archive -type f -size 0 | head -n 20
   ```

### 🖼️ Task 4.1.1 Verification: Migrated University OneDrive Archive on `UNAS 2`

**Verification Objective:** Capture a screenshot showing your `University_OneDrive_Archive` folder populated on `/mnt/unas_media` (via `du -sh` + file count, or `rclone copy` / `rsync` completion summary).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 4.1.2: Deploy FileBrowser Quantum (`homelab-gpu:8900`) Directly Over `UNAS 2`
Now let's give your `UNAS 2` RAID 1 archive a crisp, fast, self-hosted OneDrive-style web portal on `homelab-gpu` (`10.0.0.201:8900`):

1. On `homelab-gpu` (`ssh homelab-gpu`), create the FileBrowser Quantum stack and configuration file:
   ```bash
   mkdir -p ~/homelab/stacks/filebrowser/data
   cd ~/homelab/stacks/filebrowser

   cat << 'EOT' > ~/homelab/stacks/filebrowser/data/config.yaml
   server:
     port: 80
     database: "/home/filebrowser/data/filebrowser.db"
     cacheDir: "/home/filebrowser/data/tmp"
     sources:
       - path: "/nas-media"
         name: "UNAS 2 - Media & University Archive"
         config:
           defaultEnabled: true
   EOT
   ```
2. Create `~/homelab/stacks/filebrowser/docker-compose.yml`:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/filebrowser/docker-compose.yml
   name: filebrowser_quantum

   services:
     filebrowser:
       image: ghcr.io/gtsteffaniak/filebrowser:stable
       container_name: filebrowser_quantum
       ports:
         - "8900:80"
       volumes:
         - ./data:/home/filebrowser/data
         # Host-level NFSv3 automount bound cleanly into the container:
         - /mnt/unas_media:/nas-media
       environment:
         - TZ=America/New_York
         - FILEBROWSER_CONFIG=/home/filebrowser/data/config.yaml
       restart: unless-stopped
   EOT
   ```
3. Launch FileBrowser Quantum and fetch the auto-generated initial admin password from the container logs:
   ```bash
   docker compose pull
   docker compose up -d
   docker compose logs filebrowser | grep -i -E "password|admin|listening"
   ```
4. Open your browser and navigate to:
   ```text
   http://10.0.0.201:8900
   ```
   *(Or `http://homelab-gpu:8900` over Tailscale!)*
5. Sign in with `admin` and the generated password from Step 3, then **immediately rotate the admin password** under **Settings $\rightarrow$ Profile Settings**! Browse into `University_OneDrive_Archive` to confirm live file viewing, drag-and-drop uploads, and PDF/image previews work seamlessly at 2.5GbE LAN speed.

> 💡 **Optional Live Desktop Folder Sync (`Syncthing`):** If you also miss OneDrive's automatic two-way folder sync between your personal laptop and your NAS, you can optionally run **`lscr.io/linuxserver/syncthing:latest`** on `homelab-gpu` (port `8384`) mapped to `/mnt/unas_media/Personal_Files`—giving you continuous peer-to-peer sync over LAN and Tailscale with zero cloud dependency!

### 🖼️ Task 4.1.2 Verification: FileBrowser Quantum Web Portal (`10.0.0.201:8900`)

**Verification Objective:** Capture a screenshot of **FileBrowser Quantum** (`http://10.0.0.201:8900`) browsing your `UNAS 2` `MediaArchive` and migrated `University_OneDrive_Archive` folders.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Latest upstream `rclone` installed on `homelab-gpu`.
- [ ] University OneDrive files migrated onto `/mnt/unas_media/University_OneDrive_Archive` (`UNAS 2` RAID 1) and verified for file count / zero-byte integrity.
- [ ] **FileBrowser Quantum** deployed on `homelab-gpu:8900` directly over `/mnt/unas_media` with default admin password rotated.
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 4.2: OCR Document Management & Full-Text Archiving with Paperless-ngx](./04_2_document_management_paperless_ngx.ipynb).